# Part 1 – Data Audit, EDA & Business Understanding

**Dataset snapshot date:** 2025-09-30  
**Target:** `churn_next_60d` — customer made no purchase in the 60 days after the snapshot  
**Source files used:** customers, orders, support_tickets, web_events_snapshot, rfm_modeling_snapshot, intervention_history


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
SNAPSHOT = pd.Timestamp('2025-09-30')

# Data path: copy dataset CSVs into data/ subfolder, or adjust path below
DATA = Path('data')
if not DATA.exists() or not any(DATA.glob('*.csv')):
    DATA = Path('../../dataset')   # sibling dataset/ folder

CHARTS = Path('outputs/charts')
CHARTS.mkdir(parents=True, exist_ok=True)
print('Using data path:', DATA.resolve())

## 1. Load All Tables

In [ ]:
customers     = pd.read_csv(DATA / 'customers.csv')
orders        = pd.read_csv(DATA / 'orders.csv',                parse_dates=['order_date'])
tickets       = pd.read_csv(DATA / 'support_tickets.csv',       parse_dates=['ticket_date'])
web           = pd.read_csv(DATA / 'web_events_snapshot.csv')
snapshot      = pd.read_csv(DATA / 'rfm_modeling_snapshot.csv')
interventions = pd.read_csv(DATA / 'intervention_history.csv')
labels        = snapshot[['customer_id', 'churn_next_60d', 'split']].copy()

for name, df in [('customers', customers), ('orders', orders), ('tickets', tickets),
                 ('web_events', web), ('rfm_snapshot', snapshot), ('interventions', interventions)]:
    print(f'{name:20s}: {df.shape[0]:>6,} rows x {df.shape[1]} cols')

## 2. Schema Inspection

In [ ]:
for name, df in [('customers', customers), ('orders', orders), ('tickets', tickets)]:
    print(f'\n=== {name} ===')
    display(df.head(3))

## 3. Data Quality Audit

In [ ]:
# 3a. Missing values summary
print('=== Missing Values ===')
for name, df in [('customers', customers), ('orders', orders), ('tickets', tickets),
                 ('web_events', web), ('rfm_snapshot', snapshot)]:
    miss = df.isna().sum()
    miss = miss[miss > 0]
    if len(miss):
        print(f'\n{name}:')
        for col, cnt in miss.items():
            print(f'  {col}: {cnt} ({cnt/len(df)*100:.1f}%)')
    else:
        print(f'\n{name}: no missing values')

In [ ]:
# 3b. Duplicate orders (_DUP suffix) and duplicate rows
dup_orders = orders[orders['order_id'].str.endswith('_DUP', na=False)]
print(f'Orders with _DUP suffix (deduplication test rows): {len(dup_orders)}')
print('Sample:', dup_orders['order_id'].head(5).tolist())

# 3c. Post-snapshot rows (leakage risk)
post = (orders['order_date'] > SNAPSHOT).sum()
pre  = (orders['order_date'] <= SNAPSHOT).sum()
print(f'\nPre-snapshot (usable as features): {pre:,}')
print(f'Post-snapshot (label window only — DO NOT use as features): {post:,}')

In [ ]:
# 3d. Outliers in gross_amount
p99 = orders['gross_amount'].quantile(0.99)
print(f'gross_amount p99 = {p99:.0f} INR')
print(f'Orders > p99: {(orders["gross_amount"] > p99).sum()}')
display(orders['gross_amount'].describe().to_frame().T)

In [ ]:
# 3e. Join integrity
cust_ids = set(customers['customer_id'])
print('Order customer_ids not in customers:', len(set(orders['customer_id']) - cust_ids))
print('Ticket customer_ids not in customers:', len(set(tickets['customer_id']) - cust_ids))
print('Customers with NO support tickets (expected):', len(cust_ids - set(tickets['customer_id'])))
print('Web rows match customers exactly:', len(web) == len(customers))

## 4. EDA Charts (8 charts)

In [ ]:
# Chart 1: Churn label distribution
fig, ax = plt.subplots(figsize=(6, 4))
vc = labels['churn_next_60d'].value_counts().sort_index()
ax.bar(['Retained (0)', 'Churned (1)'], vc.values, color=['#4C9BE8', '#E85555'])
for i, v in enumerate(vc.values):
    ax.text(i, v + 10, f'{v}\n({v/len(labels)*100:.1f}%)', ha='center', fontsize=10)
ax.set_title('Churn Label Distribution (n=2,400)', fontweight='bold')
ax.set_ylabel('Customers')
plt.tight_layout(); plt.savefig(CHARTS / '01_churn_distribution.png', dpi=120); plt.show()

In [ ]:
# Chart 2: Churn rate by city_tier
ct = snapshot.groupby('city_tier')['churn_next_60d'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(6, 4))
ct.plot(kind='bar', ax=ax, color=['#E85555', '#F5A623', '#4C9BE8'])
ax.set_title('Churn Rate by City Tier', fontweight='bold')
ax.set_ylabel('Churn Rate'); ax.set_xticklabels(ct.index, rotation=0)
for i, v in enumerate(ct.values):
    ax.text(i, v + 0.005, f'{v:.1%}', ha='center')
plt.tight_layout(); plt.savefig(CHARTS / '02_churn_by_city_tier.png', dpi=120); plt.show()

In [ ]:
# Chart 3: Churn rate by recency bucket
snap = snapshot.copy()
snap['recency_bucket'] = pd.cut(snap['recency_days'],
    bins=[0, 30, 60, 90, 150, snap['recency_days'].max()+1],
    labels=['0-30d','31-60d','61-90d','91-150d','150d+'])
rb = snap.groupby('recency_bucket', observed=True)['churn_next_60d'].mean()
fig, ax = plt.subplots(figsize=(7, 4))
rb.plot(kind='bar', ax=ax, color='#E85555')
ax.set_title('Churn Rate by Recency Bucket', fontweight='bold')
ax.set_ylabel('Churn Rate'); ax.set_xticklabels(rb.index, rotation=0)
for i, v in enumerate(rb.values):
    ax.text(i, v + 0.01, f'{v:.1%}', ha='center', fontsize=9)
plt.tight_layout(); plt.savefig(CHARTS / '03_churn_by_recency.png', dpi=120); plt.show()

In [ ]:
# Chart 4: Churn rate by ticket count bucket
snap['ticket_bucket'] = pd.cut(snap['ticket_count_90d'],
    bins=[-1, 0, 1, 2, snap['ticket_count_90d'].max()+1],
    labels=['0 tickets','1 ticket','2 tickets','3+ tickets'])
tb = snap.groupby('ticket_bucket', observed=True)['churn_next_60d'].mean()
fig, ax = plt.subplots(figsize=(6, 4))
tb.plot(kind='bar', ax=ax, color='#F5A623')
ax.set_title('Churn Rate by Support Ticket Count (last 90d)', fontweight='bold')
ax.set_ylabel('Churn Rate'); ax.set_xticklabels(tb.index, rotation=0)
for i, v in enumerate(tb.values):
    ax.text(i, v + 0.01, f'{v:.1%}', ha='center', fontsize=9)
plt.tight_layout(); plt.savefig(CHARTS / '04_churn_by_ticket_count.png', dpi=120); plt.show()

In [ ]:
# Chart 5: Return rate vs churn  +  Discount % vs churn
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, xlabel in [
        (axes[0], 'return_rate_180d', 'Return Rate (180d)'),
        (axes[1], 'avg_discount_pct_180d', 'Avg Discount % (180d)')]:
    for cls, lbl, clr in [(0, 'Retained', '#4C9BE8'), (1, 'Churned', '#E85555')]:
        ax.hist(snap[snap['churn_next_60d']==cls][col], bins=25, alpha=0.6, label=lbl, color=clr, density=True)
    ax.set_title(f'{xlabel} vs Churn', fontweight='bold'); ax.set_xlabel(xlabel)
    ax.set_ylabel('Density'); ax.legend()
plt.tight_layout(); plt.savefig(CHARTS / '05_return_discount_vs_churn.png', dpi=120); plt.show()

In [ ]:
# Chart 6: Sessions (30d) vs churn
fig, ax = plt.subplots(figsize=(7, 4))
for cls, lbl, clr in [(0, 'Retained', '#4C9BE8'), (1, 'Churned', '#E85555')]:
    ax.hist(snap[snap['churn_next_60d']==cls]['sessions_30d'].clip(upper=20),
            bins=20, alpha=0.6, label=lbl, color=clr, density=True)
ax.set_title('Web/App Sessions (last 30d) vs Churn', fontweight='bold')
ax.set_xlabel('Sessions (capped 20)'); ax.set_ylabel('Density'); ax.legend()
plt.tight_layout(); plt.savefig(CHARTS / '06_sessions_vs_churn.png', dpi=120); plt.show()

In [ ]:
# Chart 7: Issue type distribution — churned vs retained
churned_ids  = set(labels[labels['churn_next_60d']==1]['customer_id'])
issue_df = pd.DataFrame({
    'Churned':  tickets[tickets['customer_id'].isin(churned_ids)]['issue_type'].value_counts(normalize=True),
    'Retained': tickets[~tickets['customer_id'].isin(churned_ids)]['issue_type'].value_counts(normalize=True)
}).fillna(0)
fig, ax = plt.subplots(figsize=(10, 4))
issue_df.plot(kind='bar', ax=ax, color=['#E85555','#4C9BE8'])
ax.set_title('Support Issue Type: Churned vs Retained', fontweight='bold')
ax.set_ylabel('Share of tickets'); ax.set_xticklabels(issue_df.index, rotation=30, ha='right')
plt.tight_layout(); plt.savefig(CHARTS / '07_issue_type_churned_vs_retained.png', dpi=120); plt.show()

In [ ]:
# Chart 8: Churn rate by acquisition channel
ac = snapshot.groupby('acquisition_channel')['churn_next_60d'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
ac.plot(kind='bar', ax=ax, color='#4C9BE8')
ax.set_title('Churn Rate by Acquisition Channel', fontweight='bold')
ax.set_ylabel('Churn Rate'); ax.set_xticklabels(ac.index, rotation=30, ha='right')
for i, v in enumerate(ac.values):
    ax.text(i, v + 0.005, f'{v:.1%}', ha='center', fontsize=8)
plt.tight_layout(); plt.savefig(CHARTS / '08_churn_by_channel.png', dpi=120); plt.show()

In [ ]:
# Chart 9: Intervention/campaign history vs churn
camp = snapshot[['customer_id','churn_next_60d']].merge(
    interventions[['customer_id','last_campaign_received']], on='customer_id', how='left'
)
camp['last_campaign_received'] = camp['last_campaign_received'].fillna('No Campaign')
camp_rate = camp.groupby('last_campaign_received')['churn_next_60d'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(9, 4))
camp_rate.plot(kind='bar', ax=ax, color='#7E57C2')
ax.set_title('Churn Rate by Last Campaign Received', fontweight='bold')
ax.set_ylabel('Churn Rate'); ax.set_xlabel('')
ax.set_xticklabels(camp_rate.index, rotation=30, ha='right')
for i, v in enumerate(camp_rate.values):
    ax.text(i, v + 0.005, f'{v:.1%}', ha='center', fontsize=8)
plt.tight_layout(); plt.savefig(CHARTS / '09_churn_by_last_campaign.png', dpi=120); plt.show()

## 5. Churn-Risk Hypotheses (6 hypotheses)

### H1 – High recency → elevated churn
**Evidence:** Chart 3 and the numerical table below show churn increasing from the lowest recency bucket (0-30d) to the highest (150d+).  
**Interpretation:** Customers who haven't ordered in 3+ months are disengaging — target them with reactivation triggers before the 90-day mark.

### H2 – Repeated support issues → churn precursor  
**Evidence:** Chart 4 shows 3+ ticket customers churn at significantly higher rates.  
**Interpretation:** Unresolved friction (damaged items, refund delays) erodes trust and drives exit.

### H3 – High return rate → product-fit problem → churn  
**Evidence:** Chart 5 (left) shows churned customers skew higher on return_rate_180d.  
**Interpretation:** Customers returning >30% of orders are likely dissatisfied with product fit.

### H4 – Low web/app engagement → early churn signal  
**Evidence:** Chart 6 shows churned customers have fewer sessions in the 30 days before snapshot.  
**Interpretation:** Disengagement from browsing precedes purchase dropout.

### H5 – Promo-dependency → churn when discounts disappear  
**Evidence:** Chart 5 (right) shows higher avg_discount_pct_180d in churned cohort.  
**Interpretation:** Pure deal-seekers are price-elastic and will defect when no offer is present.

### H6 – Acquisition channel affects long-run retention  
**Evidence:** Chart 8 shows variance in churn rate across channels.  
**Interpretation:** Different channels attract customers with different brand affinity levels.

### Campaign signal note (supporting evidence)  
**Evidence:** Chart 9 compares churn by last campaign received from intervention history.  
**Interpretation:** Campaign exposure quality differs; this supports channel- and segment-specific retention playbooks instead of blanket offers.

In [ ]:
# Numerical evidence for hypotheses
print('H1 – Recency quartile vs churn:')
print(snap.groupby('recency_bucket', observed=True)['churn_next_60d'].agg(['mean','count']).to_string())

print('\nH2 – Ticket cohorts:')
hi_t = snap[snap['ticket_count_90d'] >= 3]
zero_t = snap[snap['ticket_count_90d'] == 0]
print(f'  tickets >= 3: churn={hi_t["churn_next_60d"].mean():.1%} (n={len(hi_t)})')
print(f'  tickets = 0:  churn={zero_t["churn_next_60d"].mean():.1%} (n={len(zero_t)})')

print('\nH3 – Return rate cohorts:')
hi_ret = snap[snap['return_rate_180d'] > 0.3]
lo_ret = snap[snap['return_rate_180d'] == 0]
print(f'  return > 30%: churn={hi_ret["churn_next_60d"].mean():.1%} (n={len(hi_ret)})')
print(f'  return = 0%:  churn={lo_ret["churn_next_60d"].mean():.1%} (n={len(lo_ret)})')

print('\nH5 – Discount cohorts:')
hi_disc = snap[snap['avg_discount_pct_180d'] >= 0.40]
lo_disc = snap[snap['avg_discount_pct_180d'] < 0.20]
print(f'  discount >= 40%: churn={hi_disc["churn_next_60d"].mean():.1%} (n={len(hi_disc)})')
print(f'  discount < 20%:  churn={lo_disc["churn_next_60d"].mean():.1%} (n={len(lo_disc)})')

print('\nH4 – Session cohorts:')
lo_s = snap[snap['sessions_30d'] <= 2]
hi_s = snap[snap['sessions_30d'] >= 8]
print(f'  sessions <= 2: churn={lo_s["churn_next_60d"].mean():.1%} (n={len(lo_s)})')
print(f'  sessions >= 8: churn={hi_s["churn_next_60d"].mean():.1%} (n={len(hi_s)})')

print('\nH6 – Channel churn rates (top 3):')
print(snapshot.groupby('acquisition_channel')['churn_next_60d'].mean().sort_values(ascending=False).head(3).to_string())

print('\nCampaign signal – churn by last_campaign_received:')
print(camp_rate.to_string())

In [ ]:
# Export dataset-backed markdown reports required by the rubric
dq_lines = []
dq_lines.append('# Data Quality Report\n\n')
dq_lines.append(f'Snapshot date: **{SNAPSHOT.date()}**\n\n')
dq_lines.append('## 1) Missing Values\n\n')
dq_lines.append('| Table | Column | Missing Count | Missing % |\n|---|---|---:|---:|\n')
for name, df in [('customers', customers), ('orders', orders), ('support_tickets', tickets), ('web_events_snapshot', web), ('rfm_modeling_snapshot', snapshot), ('intervention_history', interventions)]:
    miss = df.isna().sum()
    miss = miss[miss > 0]
    if len(miss) == 0:
        dq_lines.append(f'| {name} | _No missing columns_ | 0 | 0.0% |\n')
    else:
        for col, cnt in miss.sort_values(ascending=False).items():
            dq_lines.append(f'| {name} | {col} | {int(cnt)} | {cnt/len(df)*100:.1f}% |\n')

dup_orders = orders['order_id'].astype(str).str.endswith('_DUP', na=False).sum()
dup_customers = customers['customer_id'].duplicated().sum()
pre = int((orders['order_date'] <= SNAPSHOT).sum())
post = int((orders['order_date'] > SNAPSHOT).sum())
p99 = orders['gross_amount'].quantile(0.99)
high_out = (orders['gross_amount'] > p99).sum()
cust_ids = set(customers['customer_id'])
dq_lines.append('\n## 2) Duplicates\n\n')
dq_lines.append(f'- Orders with `_DUP` suffix: **{int(dup_orders)}**\n')
dq_lines.append(f'- Duplicate customer IDs in `customers`: **{int(dup_customers)}**\n')
dq_lines.append('\n## 3) Date Consistency and Leakage Risk\n\n')
dq_lines.append(f'- Pre-snapshot orders (feature-safe): **{pre:,}**\n')
dq_lines.append(f'- Post-snapshot orders (must not be model features): **{post:,}**\n')
dq_lines.append('\n## 4) Outliers\n\n')
dq_lines.append(f'- `gross_amount` p99: **{p99:,.0f} INR**\n')
dq_lines.append(f'- Rows above p99: **{int(high_out)}**\n')
dq_lines.append('\n## 5) Join-Key Integrity\n\n')
dq_lines.append(f'- Order customer IDs not in customers: **{len(set(orders["customer_id"]) - cust_ids)}**\n')
dq_lines.append(f'- Ticket customer IDs not in customers: **{len(set(tickets["customer_id"]) - cust_ids)}**\n')
dq_lines.append(f'- Web customer IDs not in customers: **{len(set(web["customer_id"]) - cust_ids)}**\n')
dq_lines.append(f'- Customers with no tickets (valid state): **{len(cust_ids - set(tickets["customer_id"]))}**\n')
dq_lines.append('\n## 6) Recommended Guardrails\n\n')
dq_lines.append('- Remove `_DUP` rows before feature aggregation.\n')
dq_lines.append('- Enforce `order_date <= snapshot_date` for all feature pipelines.\n')
dq_lines.append('- Impute missing categorical values with explicit categories like `None/Unknown`.\n')
dq_lines.append('- Cap/log-transform heavy-tailed monetary values before modeling.\n')
Path('data_quality_report.md').write_text(''.join(dq_lines), encoding='utf-8')

n_total = len(snapshot)
n_churn = int(snapshot['churn_next_60d'].sum())
cr = snapshot['churn_next_60d'].mean()
rec_low = snapshot[snapshot['recency_days'] <= 30]['churn_next_60d'].mean()
rec_high = snapshot[snapshot['recency_days'] > 150]['churn_next_60d'].mean()
t0 = snapshot[snapshot['ticket_count_90d'] == 0]['churn_next_60d'].mean()
t3 = snapshot[snapshot['ticket_count_90d'] >= 3]['churn_next_60d'].mean()
r0 = snapshot[snapshot['return_rate_180d'] == 0]['churn_next_60d'].mean()
r3 = snapshot[snapshot['return_rate_180d'] > 0.30]['churn_next_60d'].mean()
s2 = snapshot[snapshot['sessions_30d'] <= 2]['churn_next_60d'].mean()
s8 = snapshot[snapshot['sessions_30d'] >= 8]['churn_next_60d'].mean()
d4 = snapshot[snapshot['avg_discount_pct_180d'] >= 0.40]['churn_next_60d'].mean()
d2 = snapshot[snapshot['avg_discount_pct_180d'] < 0.20]['churn_next_60d'].mean()
acq = snapshot.groupby('acquisition_channel')['churn_next_60d'].mean().sort_values(ascending=False)
memo = []
memo.append('# Business Memo - Churn Early Signals\n\n')
memo.append('## Context\n\n')
memo.append(f'Analyzed **{n_total:,}** customers at snapshot date **{SNAPSHOT.date()}**; churn base rate is **{cr:.1%}** ({n_churn:,} customers).\n\n')
memo.append('## Priority Findings\n\n')
memo.append(f'1. **Recency risk**: churn is **{rec_low:.1%}** for recency <= 30 days vs **{rec_high:.1%}** for recency > 150 days (Chart 3).\n')
memo.append(f'2. **Support friction**: churn is **{t0:.1%}** for 0 tickets vs **{t3:.1%}** for 3+ tickets (Charts 4 and 7).\n')
memo.append(f'3. **Return behavior**: churn is **{r0:.1%}** at 0% return rate vs **{r3:.1%}** when return_rate_180d > 30% (Chart 5 left).\n')
memo.append(f'4. **Engagement drop**: churn is **{s2:.1%}** when sessions_30d <= 2 vs **{s8:.1%}** when sessions_30d >= 8 (Chart 6).\n')
memo.append(f'5. **Discount dependency**: churn is **{d4:.1%}** for avg_discount_pct >= 40% vs **{d2:.1%}** for < 20% (Chart 5 right).\n')
memo.append(f'6. **Channel quality**: highest churn channel is **{acq.index[0]}** ({acq.iloc[0]:.1%}) vs lowest **{acq.index[-1]}** ({acq.iloc[-1]:.1%}) (Chart 8).\n\n')
memo.append('## Recommended Investigation Order\n\n')
memo.append('1. Resolve high-ticket support issues before discounting.\n')
memo.append('2. Trigger reactivation before customers cross 90-day recency.\n')
memo.append('3. Diagnose product-fit issues for high-return cohorts.\n')
memo.append('4. Use value-led messaging for discount-dependent customers.\n')
memo.append('5. Build channel-specific onboarding and retention plays.\n')
Path('business_memo.md').write_text(''.join(memo), encoding='utf-8')
print('Wrote data_quality_report.md and business_memo.md')